# kafka_spark.API.ipynb
## API Reference: Apache Kafka and Spark Structured Streaming
**Author**: Aashish Vinod  
**Course**: DATA605 Spring 2026  

This notebook covers the key APIs:
1. Kafka Producer API
2. Kafka Consumer API
3. Utility Functions API
4. Spark Structured Streaming API
5. Windowed Aggregations API

## 1. Imports

In [ ]:
import json
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from kafka import KafkaProducer, KafkaConsumer
from kafka.admin import KafkaAdminClient, NewTopic
from kafka_spark_utils import (
    generate_stock_event, generate_stock_stream,
    serialize_event, deserialize_event,
    compute_moving_average, check_price_alert,
    format_kafka_summary, STOCK_SYMBOLS, BASE_PRICES,
)
print('All imports successful!')
print(f'Stocks: {STOCK_SYMBOLS}')
print(f'Base prices: {BASE_PRICES}')


## 2. Kafka Producer API
KafkaProducer sends messages to a Kafka topic.

In [ ]:
# Initialize KafkaProducer
producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8') if k else None,
)
print(f'Producer created successfully')
print(f'Bootstrap servers: localhost:9092')


In [ ]:
# Send a single event
event = generate_stock_event('AAPL')
print(f'Event to send:')
print(json.dumps(event, indent=2))
future = producer.send('stock-prices', key='AAPL', value=event)
producer.flush()
print('Message sent successfully!')


## 3. Kafka Consumer API
KafkaConsumer reads messages from a Kafka topic.

In [ ]:
# Initialize KafkaConsumer
consumer = KafkaConsumer(
    'stock-prices',
    bootstrap_servers='localhost:9092',
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='api-demo-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    consumer_timeout_ms=3000,
)
messages = []
for msg in consumer:
    messages.append(msg.value)
consumer.close()
print(f'Total messages received: {len(messages)}')
if messages:
    print(f'Sample message: {json.dumps(messages[0], indent=2)}')


## 4. Utility Functions API

In [ ]:
# generate_stock_event(symbol)
event = generate_stock_event('MSFT')
print('generate_stock_event("MSFT"):')
print(json.dumps(event, indent=2))


In [ ]:
# compute_moving_average(prices, window)
prices = [100, 102, 101, 103, 105, 104, 106, 108, 107, 109]
ma5 = compute_moving_average(prices, window=5)
print(f'Prices:          {prices}')
print(f'MA5 (window=5):  {ma5}')


In [ ]:
# check_price_alert(price, symbol, threshold_pct)
alert1 = check_price_alert(180.0, 'AAPL', threshold_pct=1.5)
alert2 = check_price_alert(175.5, 'AAPL', threshold_pct=1.5)
print('Alert for AAPL at $180.0 (base=$175.0):')
print(json.dumps(alert1, indent=2) if alert1 else 'No alert triggered')
print('\nAlert for AAPL at $175.5 (base=$175.0):')
print(json.dumps(alert2, indent=2) if alert2 else 'No alert triggered')


In [ ]:
# serialize_event / deserialize_event
event = generate_stock_event('TSLA')
serialized = serialize_event(event)
deserialized = deserialize_event(serialized)
print(f'Original:     {event}')
print(f'Serialized:   {serialized[:60]}...')
print(f'Deserialized: {deserialized}')
print(f'Match: {event == deserialized}')


## 5. Spark Structured Streaming API

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = SparkSession.builder \
    .appName('StockMarketAPI') \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')


In [ ]:
# Define schema for stock events
stock_schema = StructType([
    StructField('symbol', StringType(), True),
    StructField('price', DoubleType(), True),
    StructField('volume', IntegerType(), True),
    StructField('timestamp', StringType(), True),
    StructField('change_pct', DoubleType(), True),
])
print('Stock event schema:')
stock_schema.printTreeString()


In [ ]:
# Read from Kafka as streaming DataFrame
kafka_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'localhost:9092') \
    .option('subscribe', 'stock-prices') \
    .option('startingOffsets', 'earliest') \
    .load()
print('Kafka streaming DataFrame schema:')
kafka_df.printSchema()


In [ ]:
# Parse JSON values and apply windowed aggregation
parsed_df = kafka_df \
    .select(F.from_json(F.col('value').cast('string'), stock_schema).alias('data')) \
    .select('data.*') \
    .withColumn('event_time', F.to_timestamp('timestamp'))

windowed = parsed_df \
    .withWatermark('event_time', '10 seconds') \
    .groupBy(F.window('event_time', '30 seconds', '10 seconds'), 'symbol') \
    .agg(
        F.avg('price').alias('avg_price'),
        F.max('price').alias('max_price'),
        F.min('price').alias('min_price'),
        F.count('price').alias('event_count'),
    )
print('Windowed aggregation schema:')
windowed.printSchema()


In [ ]:
spark.stop()
print('Spark session stopped.')
